In [1]:

import os
import joblib
import numpy as np
import pandas as pd

os.makedirs("Results/Leakage_Free", exist_ok=True)

data = joblib.load("Dataset/cycle_time_data.joblib")
df = data["df"].copy()

df["Scheduled_Start"] = pd.to_datetime(
    df["Scheduled_Start"], errors="coerce"
)
df["Actual_End"] = pd.to_datetime(
    df["Actual_End"], errors="coerce"
)

target = np.asarray(data["yraw"], dtype=float).ravel()
window = int(data["window"])
train_end = int(data["train_end"])

# Target must match historical processing time
assert np.allclose(
    target,
    df["Processing_Time"].to_numpy(dtype=float)
)

# Do not use target values from unfinished jobs
# or current/future observations
sequences = []
targets = []
indices = []
history_indices = []

for i in range(len(df)):
    forecast_time = df.iloc[i]["Scheduled_Start"]

    if pd.isna(forecast_time):
        continue

    earlier = df.iloc[:i]

    completed = earlier.loc[
        earlier["Actual_End"].notna()
        & (earlier["Actual_End"] <= forecast_time)
    ].sort_values(
        "Actual_End", kind="stable"
    )

    if len(completed) < window:
        continue

    recent = completed.tail(window)

    # Use only previously completed processing times
    sequences.append(
        recent["Processing_Time"].to_numpy(dtype=float)
    )
    targets.append(target[i])
    indices.append(i)
    history_indices.append(recent.index.to_list())

X_safe = np.asarray(sequences, dtype=np.float32)
y_safe = np.asarray(targets, dtype=np.float32)
indices_safe = np.asarray(indices, dtype=int)

train_mask = indices_safe < train_end
test_mask = indices_safe >= train_end

print("LEAKAGE-SAFE DATASET")
print("Sequences:", X_safe.shape)
print("Targets:", y_safe.shape)
print("Training samples:", train_mask.sum())
print("Testing samples:", test_mask.sum())

print("\nEvaluation periods:")
for name, (start, end) in {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995)
}.items():
    count = np.sum(
        (indices_safe >= start)
        & (indices_safe < end)
    )
    print(name, count)

# Verify every historical job was completed
# before its forecast timestamp
for i, hist in zip(indices_safe, history_indices):
    forecast_time = df.iloc[i]["Scheduled_Start"]
    assert all(
        df.loc[j, "Actual_End"] <= forecast_time
        and j < i
        for j in hist
    )

# Save corrected dataset separately
joblib.dump(
    {
        "X": X_safe,
        "y": y_safe,
        "indices": indices_safe,
        "train_end": train_end,
        "window": window,
        "history_indices": history_indices,
        "feature_names": ["Completed_Processing_Time"],
        "forecast_time": "Scheduled_Start"
    },
    "Results/Leakage_Free/corrected_sequences.joblib"
)

print("\nTemporal availability verification: PASSED")
print("Corrected dataset saved successfully.")


LEAKAGE-SAFE DATASET
Sequences: (977, 10)
Targets: (977,)
Training samples: 382
Testing samples: 595

Evaluation periods:
Stable 200
Drift 200
Recovery 195

Temporal availability verification: PASSED
Corrected dataset saved successfully.


In [2]:

# Step 16: Verify timestamps against cycle-time targets

import pandas as pd
import numpy as np

df_check = data["df"].copy()

df_check["Actual_Start"] = pd.to_datetime(
    df_check["Actual_Start"], errors="coerce"
)

df_check["Actual_End"] = pd.to_datetime(
    df_check["Actual_End"], errors="coerce"
)

# Calculate actual duration from timestamps
df_check["Timestamp_Duration"] = (
    df_check["Actual_End"] - df_check["Actual_Start"]
).dt.total_seconds() / 60

# Compare with forecasting target
df_check["Target_Cycle_Time"] = target

valid = (
    df_check["Timestamp_Duration"].notna()
    & np.isfinite(df_check["Target_Cycle_Time"])
)

difference = (
    df_check.loc[valid, "Timestamp_Duration"]
    - df_check.loc[valid, "Target_Cycle_Time"]
).abs()

print("TIMESTAMP CONSISTENCY CHECK")
print("Valid records:", valid.sum())
print("Missing timestamp records:", (~valid).sum())

print("Mean absolute difference:",
      round(difference.mean(), 3))

print("Maximum absolute difference:",
      round(difference.max(), 3))

print("Exactly matching records:",
      int(np.isclose(difference, 0).sum()))

print("\nRESULTS BY PERIOD")

for name, (start, end) in {
    "Training": (0, 400),
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995)
}.items():

    subset = df_check.iloc[start:end]

    diff = (
        subset["Timestamp_Duration"]
        - subset["Target_Cycle_Time"]
    ).abs()

    print(
        name,
        "| Valid:", diff.notna().sum(),
        "| Mean difference:", round(diff.mean(), 3)
    )


TIMESTAMP CONSISTENCY CHECK
Valid records: 867
Missing timestamp records: 128
Mean absolute difference: 10.55
Maximum absolute difference: 72.0
Exactly matching records: 528

RESULTS BY PERIOD
Training | Valid: 350 | Mean difference: 0.0
Stable | Valid: 178 | Mean difference: 0.0
Drift | Valid: 169 | Mean difference: 43.076
Recovery | Valid: 170 | Mean difference: 10.982


In [3]:

# Step 17: Correct timestamps and rebuild sequences

import pandas as pd
import numpy as np
import joblib

df_fixed = data["df"].copy()

df_fixed["Scheduled_Start"] = pd.to_datetime(
    df_fixed["Scheduled_Start"], errors="coerce"
)

df_fixed["Actual_Start"] = pd.to_datetime(
    df_fixed["Actual_Start"], errors="coerce"
)

# Reconstruct completion time from simulated target
df_fixed["Corrected_End"] = (
    df_fixed["Actual_Start"]
    + pd.to_timedelta(target, unit="m")
)

X_corrected = []
y_corrected = []
indices_corrected = []
history_corrected = []

for i in range(len(df_fixed)):

    forecast_time = df_fixed.iloc[i]["Scheduled_Start"]

    if pd.isna(forecast_time):
        continue

    earlier = df_fixed.iloc[:i]

    completed = earlier.loc[
        earlier["Corrected_End"].notna()
        & (earlier["Corrected_End"] <= forecast_time)
    ].sort_values("Corrected_End", kind="stable")

    if len(completed) < window:
        continue

    recent = completed.tail(window)

    X_corrected.append(
        recent["Processing_Time"].to_numpy(dtype=float)
    )
    y_corrected.append(target[i])
    indices_corrected.append(i)
    history_corrected.append(recent.index.to_list())

X_corrected = np.asarray(
    X_corrected, dtype=np.float32
)

y_corrected = np.asarray(
    y_corrected, dtype=np.float32
)

indices_corrected = np.asarray(
    indices_corrected, dtype=int
)

print("CORRECTED DATASET")
print("Sequences:", X_corrected.shape)
print("Targets:", y_corrected.shape)

print(
    "Training samples:",
    np.sum(indices_corrected < train_end)
)

print(
    "Testing samples:",
    np.sum(indices_corrected >= train_end)
)

print("\nEvaluation periods:")

for name, (start, end) in {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995)
}.items():
    count = np.sum(
        (indices_corrected >= start)
        & (indices_corrected < end)
    )
    print(name, count)

# Verify corrected history availability
for i, hist in zip(
    indices_corrected, history_corrected
):
    forecast_time = df_fixed.iloc[i]["Scheduled_Start"]

    assert all(
        df_fixed.loc[j, "Corrected_End"] <= forecast_time
        and j < i
        for j in hist
    )

# Save corrected data
joblib.dump(
    {
        "X": X_corrected,
        "y": y_corrected,
        "indices": indices_corrected,
        "train_end": train_end,
        "window": window,
        "history_indices": history_corrected,
        "feature_names": ["Completed_Processing_Time"],
        "forecast_time": "Scheduled_Start",
        "completion_time_method":
            "Actual_Start + simulated cycle time"
    },
    "Results/Leakage_Free/final_corrected_sequences.joblib"
)

print("\nCorrected temporal availability check: PASSED")
print("Final corrected dataset saved.")


CORRECTED DATASET
Sequences: (977, 10)
Targets: (977,)
Training samples: 382
Testing samples: 595

Evaluation periods:
Stable 200
Drift 200
Recovery 195

Corrected temporal availability check: PASSED
Final corrected dataset saved.


In [4]:

# Step 18: Prepare corrected data for LSTM

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")

# Use corrected sequences
X = X_corrected.copy()
y = y_corrected.copy()
indices = indices_corrected.copy()

train_mask = indices < train_end
test_mask = indices >= train_end

# Fit scalers on training data only
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_raw = X[train_mask]
X_test_raw = X[test_mask]

y_train_raw = y[train_mask].reshape(-1, 1)
y_test_raw = y[test_mask].reshape(-1, 1)

# Scale historical processing times
X_train = scaler_X.fit_transform(
    X_train_raw.reshape(-1, 1)
).reshape(-1, window, 1)

X_test = scaler_X.transform(
    X_test_raw.reshape(-1, 1)
).reshape(-1, window, 1)

# Scale targets
y_train = scaler_y.fit_transform(
    y_train_raw
).flatten()

y_test = scaler_y.transform(
    y_test_raw
).flatten()

test_indices = indices[test_mask]

print("Corrected LSTM dataset:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Test indices:", test_indices[0], "to", test_indices[-1])


Corrected LSTM dataset:
X_train: (382, 10, 1)
y_train: (382,)
X_test: (595, 10, 1)
y_test: (595,)
Test indices: 400 to 994


In [5]:

# Step 19: Train corrected LSTM

import os
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

os.makedirs("Results/Leakage_Free/LSTM", exist_ok=True)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)

class CorrectedLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)
        return self.fc(hidden[-1]).squeeze(-1)


lstm_model = CorrectedLSTM().to(device)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    generator=generator
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

epochs = 100
lstm_losses = []

start_time = time.time()

for epoch in range(epochs):

    lstm_model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = lstm_model(X_batch.to(device))

        loss = criterion(
            predictions,
            y_batch.to(device)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    epoch_loss = total_loss / len(train_dataset)
    lstm_losses.append(epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{epochs} "
            f"Loss: {epoch_loss:.6f}"
        )

training_time = time.time() - start_time

print("\nCorrected LSTM training completed.")
print(f"Training time: {training_time:.2f} seconds")

torch.save(
    lstm_model.state_dict(),
    "Results/Leakage_Free/LSTM/model.pth"
)

print("Corrected LSTM model saved.")


Epoch 10/100 Loss: 0.997911
Epoch 20/100 Loss: 0.990445
Epoch 30/100 Loss: 0.985249
Epoch 40/100 Loss: 0.983626
Epoch 50/100 Loss: 0.982619
Epoch 60/100 Loss: 0.977824
Epoch 70/100 Loss: 0.972539
Epoch 80/100 Loss: 0.967488
Epoch 90/100 Loss: 0.961855
Epoch 100/100 Loss: 0.955423

Corrected LSTM training completed.
Training time: 4.28 seconds
Corrected LSTM model saved.


In [6]:

# Step 19.1: Evaluate corrected LSTM

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

lstm_model.eval()

with torch.no_grad():
    pred_scaled = lstm_model(
        torch.tensor(X_test, dtype=torch.float32).to(device)
    ).cpu().numpy().reshape(-1, 1)

y_pred = scaler_y.inverse_transform(
    pred_scaled
).flatten()

y_actual = scaler_y.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

results = []

for name, (start, end) in {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995),
    "Overall": (400, 995)
}.items():

    mask = (
        (test_indices >= start) &
        (test_indices < end)
    )

    actual = y_actual[mask]
    predicted = y_pred[mask]

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    results.append({
        "Period": name,
        "Samples": len(actual),
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3)
    })

lstm_results = pd.DataFrame(results)

print("Corrected LSTM Evaluation Results:")
display(lstm_results)

lstm_results.to_csv(
    "Results/Leakage_Free/LSTM/metrics.csv",
    index=False
)

pd.DataFrame({
    "Index": test_indices,
    "Actual": y_actual,
    "Predicted": y_pred
}).to_csv(
    "Results/Leakage_Free/LSTM/predictions.csv",
    index=False
)

print("Results saved successfully.")


Corrected LSTM Evaluation Results:


,Period,Samples,MAE,RMSE
0,Stable,200,24.574,28.401
1,Drift,200,54.079,65.302
2,Recovery,195,30.426,36.092
3,Overall,595,36.409,46.168


Results saved successfully.


In [7]:

# Step 20: Train corrected iTransformer

import os
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

os.makedirs(
    "Results/Leakage_Free/iTransformer",
    exist_ok=True
)

torch.manual_seed(42)
np.random.seed(42)

class CorrectedITransformer(nn.Module):
    def __init__(
        self,
        seq_len=10,
        num_features=1,
        d_model=32,
        n_heads=4,
        n_layers=2
    ):
        super().__init__()

        self.embedding = nn.Linear(seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=64,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
            enable_nested_tensor=False
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_features * d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.embedding(x)
        x = self.encoder(x)
        return self.head(x).squeeze(-1)


itransformer_model = CorrectedITransformer(
    seq_len=window,
    num_features=1
).to(device)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    generator=generator
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    itransformer_model.parameters(),
    lr=0.001
)

epochs = 100
itransformer_losses = []

start_time = time.time()

for epoch in range(epochs):

    itransformer_model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = itransformer_model(
            X_batch.to(device)
        )

        loss = criterion(
            predictions,
            y_batch.to(device)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    epoch_loss = total_loss / len(train_dataset)
    itransformer_losses.append(epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{epochs} "
            f"Loss: {epoch_loss:.6f}"
        )

training_time = time.time() - start_time

print("\nCorrected iTransformer training completed.")
print(f"Training time: {training_time:.2f} seconds")

torch.save(
    itransformer_model.state_dict(),
    "Results/Leakage_Free/iTransformer/model.pth"
)

print("Corrected iTransformer model saved.")


Epoch 10/100 Loss: 0.860654
Epoch 20/100 Loss: 0.783984
Epoch 30/100 Loss: 0.715542
Epoch 40/100 Loss: 0.657694
Epoch 50/100 Loss: 0.612972
Epoch 60/100 Loss: 0.582025
Epoch 70/100 Loss: 0.554947
Epoch 80/100 Loss: 0.558010
Epoch 90/100 Loss: 0.532801
Epoch 100/100 Loss: 0.502273

Corrected iTransformer training completed.
Training time: 10.06 seconds
Corrected iTransformer model saved.


In [8]:

# Step 20.1: Evaluate corrected iTransformer

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Generate predictions
itransformer_model.eval()

with torch.no_grad():
    pred_scaled = itransformer_model(
        torch.tensor(
            X_test,
            dtype=torch.float32
        ).to(device)
    ).cpu().numpy().reshape(-1, 1)

# Convert predictions to original cycle-time units
y_pred_itr = scaler_y.inverse_transform(
    pred_scaled
).flatten()

y_actual_itr = scaler_y.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

# Calculate period-specific metrics
results = []

periods = {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995),
    "Overall": (400, 995)
}

for name, (start, end) in periods.items():

    mask = (
        (test_indices >= start) &
        (test_indices < end)
    )

    actual = y_actual_itr[mask]
    predicted = y_pred_itr[mask]

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    results.append({
        "Period": name,
        "Samples": len(actual),
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3)
    })

itr_results = pd.DataFrame(results)

print("Corrected iTransformer Evaluation Results:")
display(itr_results)

# Save metrics
itr_results.to_csv(
    "Results/Leakage_Free/iTransformer/metrics.csv",
    index=False
)

# Save predictions
pd.DataFrame({
    "Index": test_indices,
    "Actual": y_actual_itr,
    "Predicted": y_pred_itr
}).to_csv(
    "Results/Leakage_Free/iTransformer/predictions.csv",
    index=False
)

print("Corrected iTransformer results saved.")


Corrected iTransformer Evaluation Results:


,Period,Samples,MAE,RMSE
0,Stable,200,27.725,33.701
1,Drift,200,65.305,78.695
2,Recovery,195,33.825,41.228
3,Overall,595,42.356,54.959


Corrected iTransformer results saved.


In [9]:

# Step 21: Corrected iTransformer + ADWIN

import copy
import time
import os
import numpy as np
import pandas as pd
import torch

from river.drift import ADWIN
from torch.utils.data import TensorDataset, DataLoader

os.makedirs(
    "Results/Leakage_Free/iTransformer_ADWIN",
    exist_ok=True
)

torch.manual_seed(42)
np.random.seed(42)

# Start from the same trained iTransformer
adaptive_model = copy.deepcopy(itransformer_model).to(device)
adaptive_model.eval()

adwin = ADWIN(delta=0.002)

learning_rate = 0.0001
fine_tune_epochs = 10
adaptation_window = 100
batch_size = 16

adaptive_predictions = []
adaptive_actuals = []
adaptive_drift_points = []

def fine_tune_corrected(model, current_position):
    # Use only previously revealed labeled examples
    start = max(0, current_position - adaptation_window + 1)

    X_recent = X_test[start:current_position + 1]
    y_recent = y_test[start:current_position + 1]

    dataset = TensorDataset(
        torch.tensor(X_recent, dtype=torch.float32),
        torch.tensor(y_recent, dtype=torch.float32)
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    criterion = torch.nn.MSELoss()
    model.train()

    for epoch in range(fine_tune_epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(
                model(xb.to(device)),
                yb.to(device)
            )
            loss.backward()
            optimizer.step()

    model.eval()


start_time = time.time()

for pos in range(len(X_test)):

    x = torch.tensor(
        X_test[pos:pos+1],
        dtype=torch.float32
    ).to(device)

    adaptive_model.eval()

    with torch.no_grad():
        prediction_scaled = adaptive_model(x).cpu().numpy()

    prediction = scaler_y.inverse_transform(
        prediction_scaled.reshape(-1, 1)
    )[0, 0]

    actual = float(y_test_raw[pos, 0])
    error = abs(actual - prediction)

    adaptive_predictions.append(prediction)
    adaptive_actuals.append(actual)

    # Update detector after observing actual value
    adwin.update(error)

    if adwin.drift_detected:
        observation = int(test_indices[pos])
        adaptive_drift_points.append(observation)

        print(f"ADWIN detected change at {observation}")

        fine_tune_corrected(
            adaptive_model,
            current_position=pos
        )

execution_time = time.time() - start_time

print("\nAdaptive forecasting completed.")
print("Detected points:", adaptive_drift_points)
print("Total adaptations:", len(adaptive_drift_points))
print("Execution time:", round(execution_time, 2), "seconds")

# Save results
pd.DataFrame({
    "Index": test_indices,
    "Actual": adaptive_actuals,
    "Predicted": adaptive_predictions
}).to_csv(
    "Results/Leakage_Free/iTransformer_ADWIN/predictions.csv",
    index=False
)

torch.save(
    adaptive_model.state_dict(),
    "Results/Leakage_Free/iTransformer_ADWIN/model.pth"
)

print("Corrected adaptive predictions saved.")


ADWIN detected change at 623
ADWIN detected change at 879

Adaptive forecasting completed.
Detected points: [623, 879]
Total adaptations: 2
Execution time: 1.88 seconds
Corrected adaptive predictions saved.


In [10]:

# Step 22: Evaluate corrected iTransformer + ADWIN

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

base = pd.read_csv(
    "Results/Leakage_Free/iTransformer/predictions.csv"
)

adaptive = pd.read_csv(
    "Results/Leakage_Free/iTransformer_ADWIN/predictions.csv"
)

lstm = pd.read_csv(
    "Results/Leakage_Free/LSTM/predictions.csv"
)

comparison = (
    base[["Index", "Actual", "Predicted"]]
    .rename(columns={"Predicted": "iTransformer"})
    .merge(
        adaptive[["Index", "Actual", "Predicted"]]
        .rename(columns={
            "Actual": "Actual_ADWIN",
            "Predicted": "iTransformer_ADWIN"
        }),
        on="Index",
        validate="one_to_one"
    )
    .merge(
        lstm[["Index", "Actual", "Predicted"]]
        .rename(columns={
            "Actual": "Actual_LSTM",
            "Predicted": "LSTM"
        }),
        on="Index",
        validate="one_to_one"
    )
)

# Verify identical targets
assert len(comparison) == 595

assert np.allclose(
    comparison["Actual"],
    comparison["Actual_ADWIN"]
)

assert np.allclose(
    comparison["Actual"],
    comparison["Actual_LSTM"]
)

periods = {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995),
    "Overall": (400, 995)
}

rows = []

for name, (start, end) in periods.items():

    subset = comparison[
        (comparison["Index"] >= start) &
        (comparison["Index"] < end)
    ]

    for model_name in [
        "LSTM",
        "iTransformer",
        "iTransformer_ADWIN"
    ]:

        mae = mean_absolute_error(
            subset["Actual"],
            subset[model_name]
        )

        rmse = np.sqrt(
            mean_squared_error(
                subset["Actual"],
                subset[model_name]
            )
        )

        rows.append({
            "Period": name,
            "Model": model_name,
            "Samples": len(subset),
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3)
        })

final_results = pd.DataFrame(rows)

print("CORRECTED THREE-MODEL COMPARISON")
display(final_results)

final_results.to_csv(
    "Results/Leakage_Free/final_model_comparison.csv",
    index=False
)

print("\nFinal comparison saved successfully.")


CORRECTED THREE-MODEL COMPARISON


,Period,Model,Samples,MAE,RMSE
0,Stable,LSTM,200,24.574,28.401
1,Stable,iTransformer,200,27.725,33.701
2,Stable,iTransformer_ADWIN,200,27.725,33.701
3,Drift,LSTM,200,54.079,65.302
4,Drift,iTransformer,200,65.305,78.695
5,Drift,iTransformer_ADWIN,200,57.501,69.888
6,Recovery,LSTM,195,30.426,36.092
7,Recovery,iTransformer,195,33.825,41.228
8,Recovery,iTransformer_ADWIN,195,28.756,35.198
9,Overall,LSTM,595,36.409,46.168



Final comparison saved successfully.
